# 15 · Paddy Doctor — cinco nuevos emparejamientos de P5
**Julián Coronel Reyes · Refuerzo del manuscrito · versión 1.0.0**

Este cuaderno ejecuta **120 entrenamientos nuevos y 240 evaluaciones**: 5 construcciones × 4 arquitecturas × 3 semillas × 2 dominios. No usa como inicialización los modelos Paddy Doctor anteriores: cada corrida parte de pesos preentrenados identificados.

**Para iniciar:** selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU** (L4 si está disponible) y **Ejecutar todas**. Autoriza Drive. La ruta predeterminada es `MyDrive/PADDY_DOCTOR_Q1`. Si tus imágenes ya están en otro lugar, indícalo en `DATASET_ROOT` en la celda 2; de lo contrario se intentará obtener el dataset con Kaggle.

**Tiempo orientativo:** reservar 6–8 horas en una L4, basado en el entrenamiento histórico; descargas y escritura de checkpoints pueden ampliarlo. Se actualiza una estimación con los tiempos reales. **Espacio:** reserva unos 15 GB libres en Drive para los 120 mejores checkpoints y la caché. En CPU el entrenamiento se detiene antes de empezar.

**Si se corta Colab:** vuelve a ejecutar todas las celdas con el mismo nombre del experimento y entorno. Se verifican y omiten corridas completas. La corrida que quedó a medias se reinicia desde sus pesos preentrenados; no continúa desde una época intermedia. No ejecutes dos sesiones a la vez sobre la misma carpeta.

El cuaderno incorpora los metadatos near-clean congelados y la asignación P5 original para medir solapamiento. **No incorpora imágenes ni credenciales.** Las nuevas construcciones pueden compartir imágenes: no son cinco cohortes de campo independientes.


## 1. Conectar Google Drive
No se modifican resultados ni modelos de los notebooks 00–14. Todo el nuevo experimento se guarda en carpetas que empiezan por `15_`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, shutil, time

drive.mount('/content/drive')


## 2. Configuración de ejecución
Puedes cambiar las rutas si lo necesitas. Conserva el diseño científico para las 120 corridas. `NUM_WORKERS=2` es el valor inicial; si tu runtime falla al crear procesos, usa 0 antes de empezar un experimento nuevo. Un cambio de configuración o entorno no se mezcla automáticamente con resultados anteriores.


In [ ]:
PROJECT = Path('/content/drive/MyDrive/PADDY_DOCTOR_Q1')
EXPERIMENT_NAME = '15_P5_Rematching_5x3_30ep_v1'
DATASET_ROOT = ''  # Opcional: ruta que contiene train_images/ o la propia carpeta train_images/
NUM_WORKERS = 2

RUN_ROOT = PROJECT / 'results' / EXPERIMENT_NAME
MODEL_ROOT = PROJECT / 'models' / EXPERIMENT_NAME
DESIGN_ROOT = PROJECT / 'data' / 'splits' / EXPERIMENT_NAME
EXPORTS_DIR = PROJECT / 'exports'
for p in [RUN_ROOT, MODEL_ROOT, DESIGN_ROOT, EXPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Resultados:', RUN_ROOT)
print('Checkpoints:', MODEL_ROOT)


## 3. Preparar dependencias
Se conserva el par PyTorch/torchvision suministrado por Colab y se fija `timm==1.0.28`. Se registran versiones exactas y GPU al iniciar. Una reanudación con versiones o GPU distintas se detiene para evitar mezclar entornos; para retomar, selecciona la misma GPU y restaura las versiones del archivo `environment_lock.json`.


In [ ]:
# No se instala ni reemplaza manualmente el par torch/torchvision de Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.28', 'kagglehub', 'pandas', 'scipy',
                'scikit-learn', 'matplotlib', 'Pillow'], check=True)


## 4. Restaurar código y metadatos congelados
Las siguientes cadenas comprimidas son datos y código del cuaderno. Se verifican sus huellas antes de usarlos. La copia de metadatos queda dentro del nuevo experimento; el CSV histórico no se sobrescribe.


In [ ]:
import base64, zlib, hashlib, importlib.util
MODULE_TEXT = '"""Paddy Doctor P5 repeated matching, version 1.0.0.\nFresh network fits; frozen class/age matching; source-validation model selection.\nNo claim of independent acquisition cohorts or external validation.\n"""\nfrom pathlib import Path\nimport os, sys, json, hashlib, time, random, gc, shutil, platform, itertools\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax, logsumexp\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix\n\nVERSION = \'1.0.0\'\nCLASSES = [\'blast\', \'dead_heart\', \'downy_mildew\', \'hispa\', \'normal\']\nDOMAINS = [\'ADT45\', \'OTHER\']\nMATCH_SEEDS = [31001, 31002, 31003, 31004, 31005]\nTRAIN_SEEDS = [2026, 2027, 2028]\nMODELS = {\n    \'resnet50\': \'resnet50.a1_in1k\',\n    \'convnext_tiny\': \'convnext_tiny.in12k_ft_in1k\',\n    \'vit_small_patch16_224\': \'vit_small_patch16_224.augreg_in21k_ft_in1k\',\n    \'swin_tiny_patch4_window7_224\': \'swin_tiny_patch4_window7_224.ms_in1k\',\n}\nRECIPE = dict(epochs=30, image_size=224, resize_size=256, batch_size=32,\n              lr=3e-4, weight_decay=.05, warmup_epochs=2,\n              warmup_start_factor=.1, cosine_eta_min=3e-6,\n              crop_scale=[.85, 1.], crop_ratio=[.75, 4/3], hflip=.5,\n              rotation_degrees=10, jitter=[.1, .1, .1, .02],\n              mean=[.485, .456, .406], std=[.229, .224, .225],\n              early_stopping=False, selection=\'source_validation_macro_f1\',\n              improvement_tolerance=1e-6, amp=\'fp16_on_cuda\',\n              loader_randomness=\'deterministic per-epoch seed; workers recreated each epoch\',\n              resume=\'completed runs skipped; interrupted run restarts from pretrained initialization\')\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_bytes(path, payload):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    with open(tmp, \'wb\') as f:\n        f.write(payload); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef write_json(path, obj):\n    atomic_bytes(path, (json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, default=str)+\'\\n\').encode())\n\n\ndef write_csv(path, df):\n    atomic_bytes(path, df.to_csv(index=False, lineterminator=\'\\n\').encode())\n\n\ndef freeze_json(path, obj):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != obj:\n            raise RuntimeError(f\'Cambió la configuración o el entorno: {path}. No mezclar resultados. Usa otro EXPERIMENT_NAME para un experimento diferente.\')\n    else: write_json(path, obj)\n\n\ndef load_metadata(path, expected_n=10329):\n    d = pd.read_csv(path, dtype={\'image_id\':str, \'label\':str, \'variety\':str, \'sha256\':str})\n    required={\'image_id\',\'label\',\'variety\',\'age\',\'sha256\'}\n    if not required.issubset(d): raise ValueError(f\'Metadata sin columnas: {required-set(d)}\')\n    if expected_n is not None and len(d)!=expected_n: raise ValueError(f\'Se esperaban {expected_n} imágenes, hay {len(d)}\')\n    if d[list(required)].isna().any().any() or d.image_id.duplicated().any(): raise ValueError(\'Metadata incompleta o IDs repetidos\')\n    if d.sha256.duplicated().any(): raise ValueError(\'Hay duplicados exactos en el pool near-clean\')\n    age=pd.to_numeric(d.age, errors=\'raise\')\n    if not np.isfinite(age).all() or (age!=np.floor(age)).any(): raise ValueError(\'Edad debe ser entera\')\n    d[\'age\']=age.astype(int)\n    if any(Path(x).name!=x for x in d.image_id): raise ValueError(\'image_id debe ser solo un nombre de archivo\')\n    d[\'domain\']=np.where(d.variety.eq(\'ADT45\'),\'ADT45\',\'OTHER\')\n    return d.sort_values(\'image_id\').reset_index(drop=True)\n\n\ndef construct_cohort(meta, match_seed):\n    """New pairing without replacement within cohort; class-stratified pair split 60/20/20."""\n    rng=np.random.default_rng(match_seed); records=[]\n    support=meta.groupby([\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).reindex(columns=DOMAINS,fill_value=0)\n    eligible=support.min(axis=1).groupby(level=0).sum()\n    if sorted(eligible[eligible>=100].index)!=CLASSES: raise ValueError(\'Las clases elegibles no coinciden con el P5 congelado\')\n    for label in CLASSES:\n        candidates=[]\n        sub=meta[meta.label.eq(label)]\n        for age in sorted(sub.age.unique()):\n            pools=[sub[sub.age.eq(age)&sub.domain.eq(domain)].sort_values(\'image_id\') for domain in DOMAINS]\n            pools=[p.iloc[rng.permutation(len(p))].reset_index(drop=True) for p in pools]\n            for i in range(min(map(len,pools))):\n                candidates.append(dict(label=label, age=int(age), ADT45_image_id=pools[0].iloc[i].image_id,\n                                       OTHER_image_id=pools[1].iloc[i].image_id))\n        chosen=[candidates[i] for i in rng.choice(len(candidates),100,replace=False)]\n        chosen=sorted(chosen,key=lambda r:(r[\'age\'],r[\'ADT45_image_id\'],r[\'OTHER_image_id\']))\n        order=rng.permutation(100); split=np.empty(100,dtype=object)\n        split[order[:60]]=\'train\';split[order[60:80]]=\'validation\';split[order[80:]]=\'test\'\n        for i,r in enumerate(chosen):\n            r.update(matched_pair_id=f\'M{match_seed}_{label}_{i:03d}\',pair_split=split[i],match_seed=int(match_seed))\n            records.append(r)\n    pairs=pd.DataFrame(records);lookup=meta.set_index(\'image_id\');rows=[]\n    for r in records:\n        for domain in DOMAINS:\n            image_id=r[f\'{domain}_image_id\'];m=lookup.loc[image_id]\n            rows.append(dict(image_id=image_id,label=r[\'label\'],age=r[\'age\'],variety=m.variety,\n                             domain=domain,sha256=m.sha256,matched_pair_id=r[\'matched_pair_id\'],\n                             pair_split=r[\'pair_split\'],match_seed=int(match_seed),\n                             visual_group_id=m.get(\'visual_group_id\',None)))\n    long=pd.DataFrame(rows).sort_values([\'domain\',\'pair_split\',\'label\',\'image_id\']).reset_index(drop=True)\n    validate_cohort(long)\n    return pairs,long\n\n\ndef validate_cohort(long):\n    if len(long)!=1000 or long.image_id.nunique()!=1000: raise ValueError(\'P5 debe tener 1000 imágenes únicas\')\n    if long.sha256.nunique()!=1000: raise ValueError(\'Duplicados SHA dentro de la cohorte\')\n    for split,n in [(\'train\',60),(\'validation\',20),(\'test\',20)]:\n        sub=long[long.pair_split.eq(split)]\n        counts=sub.groupby([\'domain\',\'label\']).size().reindex(pd.MultiIndex.from_product([DOMAINS,CLASSES]),fill_value=0)\n        if not counts.eq(n).all(): raise ValueError(\'Tamaño por clase/dominio incorrecto\')\n        age=sub.groupby([\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).reindex(columns=DOMAINS,fill_value=0)\n        if not age.ADT45.eq(age.OTHER).all(): raise ValueError(\'Edad no coincide exactamente entre dominios\')\n    if long.groupby(\'matched_pair_id\').size().ne(2).any(): raise ValueError(\'Cada par debe contener dos imágenes\')\n    for col in [\'pair_split\',\'label\',\'age\']:\n        if long.groupby(\'matched_pair_id\')[col].nunique().ne(1).any(): raise ValueError(f\'Par incoherente: {col}\')\n    if \'visual_group_id\' in long:\n        grouped=long.dropna(subset=[\'visual_group_id\']).groupby(\'visual_group_id\').pair_split.nunique()\n        if grouped.gt(1).any(): raise ValueError(\'Grupo visual cruza particiones: revisar pool near-clean\')\n\n\ndef prepare_design(meta, output, match_seeds=MATCH_SEEDS, reference=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True);cohorts={};rows=[]\n    for seed in match_seeds:\n        pairs,long=construct_cohort(meta,seed);cohorts[seed]=long\n        folder=output/f\'match_{seed}\';folder.mkdir(exist_ok=True)\n        for name,df in [(\'pairs.csv\',pairs),(\'assignments.csv\',long)]:\n            payload=df.to_csv(index=False,lineterminator=\'\\n\').encode();p=folder/name\n            if p.exists() and p.read_bytes()!=payload: raise RuntimeError(f\'Asignaciones congeladas diferentes: {p}\')\n            if not p.exists():atomic_bytes(p,payload)\n        rows.append(dict(match_seed=seed,n_images=1000,n_pairs=500,assignments_sha256=sha(folder/\'assignments.csv\')))\n    comparison=dict(cohorts)\n    if reference is not None:comparison[\'original_2026\']=reference\n    overlap=[]\n    for (a,da),(b,db) in itertools.combinations(comparison.items(),2):\n        for split in [\'all\',\'train\',\'validation\',\'test\']:\n            aa=set(da.image_id if split==\'all\' else da.loc[da.pair_split.eq(split),\'image_id\'])\n            bb=set(db.image_id if split==\'all\' else db.loc[db.pair_split.eq(split),\'image_id\'])\n            overlap.append(dict(cohort_a=a,cohort_b=b,partition=split,shared_images=len(aa&bb),union_images=len(aa|bb),jaccard=len(aa&bb)/len(aa|bb)))\n        for sa,sb in [(\'train\',\'test\'),(\'test\',\'train\')]:\n            aa=set(da.loc[da.pair_split.eq(sa),\'image_id\']);bb=set(db.loc[db.pair_split.eq(sb),\'image_id\'])\n            overlap.append(dict(cohort_a=a,cohort_b=b,partition=f\'{sa}_vs_{sb}\',shared_images=len(aa&bb),union_images=len(aa|bb),jaccard=len(aa&bb)/len(aa|bb)))\n    write_csv(output/\'overlap_between_cohorts.csv\',pd.DataFrame(overlap))\n    write_csv(output/\'design_manifest.csv\',pd.DataFrame(rows))\n    write_csv(output/\'candidate_support.csv\',meta.groupby([\'label\',\'age\',\'domain\']).size().rename(\'n\').reset_index())\n    return cohorts\n\n\ndef calibration_errors(y,p,bins=15,adaptive=False):\n    confidence=p.max(1); correct=(p.argmax(1)==y).astype(float)\n    edges=np.unique(np.quantile(confidence,np.linspace(0,1,bins+1))) if adaptive else np.linspace(0,1,bins+1)\n    if len(edges)<2:return float(abs(correct.mean()-confidence.mean())),float(abs(correct.mean()-confidence.mean()))\n    errors=[];weights=[]\n    for j in range(len(edges)-1):\n        use=(confidence>=edges[j] if j==0 else confidence>edges[j]) & (confidence<=edges[j+1])\n        if use.any():errors.append(abs(correct[use].mean()-confidence[use].mean()));weights.append(use.mean())\n    return float(np.dot(errors,weights)),float(max(errors,default=0.))\n\n\ndef score(y,logits,temperature=1.):\n    p=softmax(logits/temperature,axis=1);pred=p.argmax(1)\n    ece,mce=calibration_errors(y,p);adaptive,_=calibration_errors(y,p,10,True)\n    return dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),\n                macro_f1=float(f1_score(y,pred,labels=range(len(CLASSES)),average=\'macro\',zero_division=0)),\n                nll=float(np.mean(logsumexp(logits/temperature,axis=1)-(logits/temperature)[np.arange(len(y)),y])),\n                brier=float(np.mean(np.sum((p-np.eye(len(CLASSES))[y])**2,axis=1))),\n                ece_15=ece,adaptive_ece_10=adaptive,mce_15=mce)\n\n\ndef fit_temperature(y,logits):\n    result=minimize_scalar(lambda u:score(y,logits,np.exp(u))[\'nll\'],method=\'bounded\',\n                           bounds=(np.log(.05),np.log(20)),options={\'xatol\':1e-5})\n    if not result.success or not np.isfinite(result.fun):raise RuntimeError(\'Falló el ajuste de temperatura fuente\')\n    return float(np.exp(result.x)),dict(success=True,log_temperature=float(result.x),source_nll=float(result.fun),nfev=int(result.nfev))\n\n\ndef load_torch():\n    global torch,timm,T,InterpolationMode,Image,DataLoader\n    import torch,timm\n    from torchvision import transforms as T\n    from torchvision.transforms import InterpolationMode\n    from PIL import Image\n    from torch.utils.data import DataLoader\n\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n\n\ndef state_hash(state):\n    h=hashlib.sha256()\n    for name,t in sorted(state.items()):\n        t=t.detach().cpu().contiguous()\n        h.update(name.encode());h.update(str(t.dtype).encode());h.update(str(tuple(t.shape)).encode())\n        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())\n    return h.hexdigest()\n\n\ndef atomic_torch(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\');torch.save(obj,tmp);os.replace(tmp,path)\n\n\ndef environment():\n    import importlib.metadata\n    return dict(python=platform.python_version(),packages={x:importlib.metadata.version(x) for x in\n                [\'torch\',\'torchvision\',\'timm\',\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'Pillow\']},\n                cuda=str(torch.version.cuda),cudnn=int(torch.backends.cudnn.version() or 0),\n                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\')\n\n\ndef ensure_pretrained(cache,models=MODELS):\n    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)\n    for key,tag in models.items():\n        p=cache/(key+\'.pt\');manifest=cache/(key+\'.json\')\n        if p.exists() and manifest.exists():\n            info=json.loads(manifest.read_text())\n            if info[\'tag\']!=tag or sha(p)!=info[\'file_sha256\']:raise RuntimeError(\'Caché de pesos inconsistente\')\n            print(\'Pesos verificados:\',tag);continue\n        print(\'Descargando una vez:\',tag,flush=True)\n        seed_all(0)\n        model=timm.create_model(tag,pretrained=True,num_classes=1000).cpu()\n        state={k:v.detach().cpu() for k,v in model.state_dict().items()}\n        cfg=json.loads(json.dumps(model.pretrained_cfg,default=str))\n        atomic_torch(p,state)\n        write_json(manifest,dict(tag=tag,file_sha256=sha(p),tensor_sha256=state_hash(state),pretrained_cfg=cfg))\n        del model,state;gc.collect()\n\n\ndef transforms(training):\n    if training:\n        return T.Compose([T.Resize(256,interpolation=InterpolationMode.BICUBIC),\n            T.RandomResizedCrop(224,scale=(.85,1.),ratio=(.75,4/3),interpolation=InterpolationMode.BICUBIC),\n            T.RandomHorizontalFlip(.5),T.RandomRotation(10,interpolation=InterpolationMode.BILINEAR),\n            T.ColorJitter(brightness=.1,contrast=.1,saturation=.1,hue=.02),T.ToTensor(),T.Normalize(RECIPE[\'mean\'],RECIPE[\'std\'])])\n    return T.Compose([T.Resize(256,interpolation=InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize(RECIPE[\'mean\'],RECIPE[\'std\'])])\n\n\nclass ImageDataset:\n    def __init__(self,df,lookup,training=False):\n        self.df=df.reset_index(drop=True);self.lookup=lookup;self.transform=transforms(training)\n    def __len__(self):return len(self.df)\n    def __getitem__(self,i):\n        r=self.df.iloc[i]\n        with Image.open(self.lookup[r.image_id]) as im:x=self.transform(im.convert(\'RGB\'))\n        return x,CLASSES.index(r.label)\n\n\ndef worker_seed(worker_id):\n    seed=torch.initial_seed()%(2**32);np.random.seed(seed);random.seed(seed)\n\n\ndef loader(df,lookup,seed,training,workers=2):\n    generator=torch.Generator().manual_seed(seed)\n    return DataLoader(ImageDataset(df,lookup,training),batch_size=RECIPE[\'batch_size\'],shuffle=training,\n                      generator=generator,num_workers=workers,pin_memory=torch.cuda.is_available(),drop_last=False,\n                      persistent_workers=False,worker_init_fn=worker_seed)\n\n\ndef predict(model,data_loader,device):\n    model.eval();ys=[];zs=[]\n    with torch.no_grad():\n        for x,y in data_loader:\n            with torch.amp.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                z=model(x.to(device,non_blocking=True))\n            ys.append(y.numpy());zs.append(z.float().cpu().numpy().astype(np.float64))\n    return np.concatenate(ys),np.concatenate(zs)\n\n\ndef fit(model,train_df,val_df,lookup,seed,device,history_path,workers=2,epochs=None):\n    epochs=epochs or RECIPE[\'epochs\'];optimizer=torch.optim.AdamW(model.parameters(),lr=RECIPE[\'lr\'],weight_decay=RECIPE[\'weight_decay\'])\n    warm=torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=.1,end_factor=1.,total_iters=2)\n    cosine=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,epochs-2),eta_min=3e-6)\n    scheduler=torch.optim.lr_scheduler.SequentialLR(optimizer,[warm,cosine],milestones=[2])\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\');criterion=torch.nn.CrossEntropyLoss()\n    best=-np.inf;best_epoch=None;best_state=None;history=[]\n    val_loader=loader(val_df,lookup,seed,False,workers)\n    for epoch in range(1,epochs+1):\n        epoch_seed=(int(seed)*100003+epoch)%(2**32)\n        seed_all(epoch_seed)\n        train_loader=loader(train_df,lookup,epoch_seed,True,workers)\n        start=time.perf_counter();model.train();loss_sum=0.;n=0\n        for x,y in train_loader:\n            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True);optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                logits=model(x);loss=criterion(logits,y)\n            if not torch.isfinite(loss):raise RuntimeError(\'Loss no finita; se detiene sin marcar COMPLETE\')\n            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n            loss_sum+=float(loss.detach())*len(y);n+=len(y)\n        yv,zv=predict(model,val_loader,device);metrics=score(yv,zv)\n        if not np.isfinite(zv).all():raise RuntimeError(\'Logits de validación no finitos\')\n        if metrics[\'macro_f1\']>best+RECIPE[\'improvement_tolerance\']:\n            best=metrics[\'macro_f1\'];best_epoch=epoch\n            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        history.append(dict(epoch=epoch,train_loss=loss_sum/n,val_loss=metrics[\'nll\'],val_macro_f1=metrics[\'macro_f1\'],\n                            val_accuracy=metrics[\'accuracy\'],learning_rate=optimizer.param_groups[0][\'lr\'],epoch_seconds=time.perf_counter()-start))\n        write_csv(history_path,pd.DataFrame(history));scheduler.step()\n        if epoch==1 or epoch%5==0:print(f\'  época {epoch:02}/{epochs}: val F1={metrics["macro_f1"]:.4f}; mejor={best:.4f}\',flush=True)\n    model.load_state_dict(best_state)\n    return dict(best_epoch=best_epoch,best_val_macro_f1=best,epochs_completed=epochs),best_state\n\n\ndef export_predictions(model,df,lookup,seed,device,destination,temperature,workers=2):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    y,z=predict(model,loader(df,lookup,seed,False,workers),device)\n    if not np.isfinite(z).all():raise RuntimeError(\'Logits no finitos\')\n    raw=softmax(z,axis=1);scaled=softmax(z/temperature,axis=1)\n    data=df.reset_index(drop=True).copy();data[\'y_true_idx\']=y\n    for name,p in [(\'raw\',raw),(\'TS\',scaled)]:\n        data[f\'y_pred_{name}\']=[CLASSES[i] for i in p.argmax(1)];data[f\'confidence_{name}\']=p.max(1)\n        for i,label in enumerate(CLASSES):data[f\'prob_{name}__{label}\']=p[:,i]\n    for i,label in enumerate(CLASSES):data[f\'logit__{label}\']=z[:,i]\n    write_csv(destination/\'predictions.csv\',data)\n    np.savez_compressed(destination/\'logits.npz\',y_true=y,logits=z,image_id=df.image_id.to_numpy(dtype=str),class_names=np.array(CLASSES))\n    precision,recall,f1,support=precision_recall_fscore_support(y,raw.argmax(1),labels=range(5),zero_division=0)\n    write_csv(destination/\'per_class.csv\',pd.DataFrame(dict(class_name=CLASSES,precision=precision,recall=recall,f1=f1,support=support)))\n    cm=pd.DataFrame(confusion_matrix(y,raw.argmax(1),labels=range(5)),columns=CLASSES);cm.insert(0,\'true_class\',CLASSES)\n    write_csv(destination/\'confusion_counts.csv\',cm)\n    metrics=dict(raw=score(y,z),TS=score(y,z,temperature));write_json(destination/\'metrics.json\',metrics)\n    return metrics\n\n\ndef prepare_images(meta,cohorts,image_root,cache_root):\n    from PIL import Image\n    image_root=Path(image_root);cache_root=Path(cache_root);cache_root.mkdir(parents=True,exist_ok=True)\n    selected=set().union(*(set(d.image_id) for d in cohorts.values()))\n    lookup={};metadata=meta.set_index(\'image_id\');rows=[]\n    for i,image_id in enumerate(sorted(selected),1):\n        row=metadata.loc[image_id];source=image_root/row.label/image_id;dest=cache_root/row.label/image_id\n        if not dest.exists():\n            if not source.is_file():raise FileNotFoundError(f\'No se encuentra {source}\')\n            dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,dest)\n        digest=sha(dest)\n        if digest!=row.sha256:raise ValueError(f\'Los bytes de {image_id} no coinciden con el pool auditado. No se entrena.\')\n        with Image.open(dest) as image:image.verify()\n        lookup[image_id]=str(dest);rows.append(dict(image_id=image_id,sha256=digest))\n        if i%500==0:print(f\'  imágenes verificadas: {i}/{len(selected)}\',flush=True)\n    return lookup,pd.DataFrame(rows)\n\n\ndef complete_ok(run_dir,signature,model_dir):\n    marker=Path(run_dir)/\'COMPLETE.json\'\n    if not marker.exists():return False\n    payload=json.loads(marker.read_text())\n    if payload[\'signature\']!=signature:raise RuntimeError(\'COMPLETE corresponde a otra configuración\')\n    for item in payload[\'files\']:\n        base=Path(model_dir) if item[\'area\']==\'models\' else Path(run_dir)\n        p=base/item[\'path\']\n        if not p.is_file() or sha(p)!=item[\'sha256\']:raise RuntimeError(f\'Corrida marcada completa pero archivo ausente/cambiado: {p}\')\n    return True\n\n\ndef run_one(cohort,match_seed,key,train_seed,train_domain,lookup,root,model_root,weight_cache,signature,workers=2,device=None):\n    run_name=f\'match_{match_seed}/train_{train_domain}__{key}__seed_{train_seed}\'\n    run_dir=Path(root)/\'runs\'/run_name;model_dir=Path(model_root)/run_name\n    if complete_ok(run_dir,signature,model_dir):print(\'Completado, se omite:\',run_name,flush=True);return\n    run_dir.mkdir(parents=True,exist_ok=True);model_dir.mkdir(parents=True,exist_ok=True)\n    print(\'\\nENTRENANDO\',run_name,flush=True);seed_all(train_seed)\n    start=time.perf_counter();device=torch.device(device or \'cuda\')\n    model=timm.create_model(MODELS[key],pretrained=False,num_classes=1000)\n    model.load_state_dict(torch.load(Path(weight_cache)/(key+\'.pt\'),map_location=\'cpu\',weights_only=True))\n    model.reset_classifier(len(CLASSES))\n    initialization_hash=state_hash(model.state_dict())\n    model=model.to(device)\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n    train=cohort[cohort.domain.eq(train_domain)&cohort.pair_split.eq(\'train\')].reset_index(drop=True)\n    val=cohort[cohort.domain.eq(train_domain)&cohort.pair_split.eq(\'validation\')].reset_index(drop=True)\n    provenance=dict(signature=signature,match_seed=match_seed,model_key=key,tag=MODELS[key],train_seed=train_seed,\n        train_domain=train_domain,initialized_tensor_sha256=initialization_hash,\n        pretrained=json.loads((Path(weight_cache)/(key+\'.json\')).read_text()),\n        n_parameters=sum(p.numel() for p in model.parameters()),recipe=RECIPE)\n    write_json(run_dir/\'provenance.json\',provenance)\n    best,best_state=fit(model,train,val,lookup,train_seed,device,run_dir/\'history.csv\',workers)\n    checkpoint=model_dir/\'best_model.pt\'\n    atomic_torch(checkpoint,dict(model_state_dict=best_state,provenance=provenance,best=best,class_names=CLASSES))\n    yv,zv=predict(model,loader(val,lookup,train_seed,False,workers),device)\n    temperature,diagnostics=fit_temperature(yv,zv)\n    write_json(run_dir/\'temperature.json\',dict(temperature=temperature,fit_domain=train_domain,diagnostics=diagnostics,\n               note=\'source validation reused for checkpoint and temperature; no formal calibration guarantee\'))\n    export_predictions(model,val,lookup,train_seed,device,run_dir/\'source_validation\',temperature,workers)\n    evaluations=[]\n    for target in DOMAINS:\n        test=cohort[cohort.domain.eq(target)&cohort.pair_split.eq(\'test\')].reset_index(drop=True)\n        metrics=export_predictions(model,test,lookup,train_seed,device,run_dir/f\'eval__{target}\',temperature,workers)\n        evaluations.append(dict(match_seed=match_seed,model_key=key,train_seed=train_seed,train_domain=train_domain,\n                                eval_domain=target,**{f\'{cal}_{name}\':value for cal,m in metrics.items() for name,value in m.items()}))\n    write_csv(run_dir/\'evaluations.csv\',pd.DataFrame(evaluations))\n    write_json(run_dir/\'training_summary.json\',dict(**best,wall_seconds=time.perf_counter()-start,peak_gpu_memory_gb=torch.cuda.max_memory_allocated()/1024**3 if device.type==\'cuda\' else 0.))\n    files=[]\n    for area,folder in [(\'results\',run_dir),(\'models\',model_dir)]:\n        for p in sorted(folder.rglob(\'*\')):\n            if p.is_file() and p.name!=\'COMPLETE.json\' and not p.name.endswith(\'.tmp\'):\n                files.append(dict(area=area,path=p.relative_to(folder).as_posix(),sha256=sha(p)))\n    write_json(run_dir/\'COMPLETE.json\',dict(signature=signature,files=files))\n    del model,best_state;gc.collect();torch.cuda.empty_cache()\n\n\ndef summarize(root,expected_runs=120):\n    root=Path(root);parts=[]\n    for marker in sorted((root/\'runs\').rglob(\'COMPLETE.json\')):parts.append(pd.read_csv(marker.parent/\'evaluations.csv\'))\n    if not parts:raise RuntimeError(\'Todavía no hay corridas completas\')\n    evaluations=pd.concat(parts,ignore_index=True);write_csv(root/\'evaluations_all.csv\',evaluations)\n    rows=[]\n    for (match,key,seed,target),g in evaluations.groupby([\'match_seed\',\'model_key\',\'train_seed\',\'eval_domain\']):\n        control=g[g.train_domain.eq(target)];transfer=g[~g.train_domain.eq(target)]\n        if len(control)!=1 or len(transfer)!=1:continue\n        rows.append(dict(match_seed=match,model_key=key,train_seed=seed,target=target,\n                         control_f1=float(control.raw_macro_f1.iloc[0]),transfer_f1=float(transfer.raw_macro_f1.iloc[0]),\n                         loss=float(control.raw_macro_f1.iloc[0]-transfer.raw_macro_f1.iloc[0])))\n    losses=pd.DataFrame(rows)\n    if losses.empty:raise RuntimeError(\'Aún falta el dominio pareado para comparar el mismo objetivo\')\n    write_csv(root/\'fixed_target_losses_by_seed.csv\',losses)\n    complete=(losses.groupby([\'match_seed\',\'model_key\',\'target\']).train_seed.nunique()==3)\n    cohort=losses.groupby([\'match_seed\',\'model_key\',\'target\']).agg(n_seeds=(\'train_seed\',\'nunique\'),\n        loss_mean=(\'loss\',\'mean\'),loss_sd_seeds=(\'loss\',\'std\'),loss_min_seed=(\'loss\',\'min\'),loss_max_seed=(\'loss\',\'max\'),\n        control_f1=(\'control_f1\',\'mean\'),transfer_f1=(\'transfer_f1\',\'mean\')).reset_index()\n    write_csv(root/\'fixed_target_losses_by_cohort.csv\',cohort)\n    full=cohort[cohort.n_seeds.eq(3)]\n    across=full.groupby([\'model_key\',\'target\']).agg(n_rematchings=(\'match_seed\',\'nunique\'),loss_mean=(\'loss_mean\',\'mean\'),\n        loss_sd_between_rematchings=(\'loss_mean\',\'std\'),loss_min=(\'loss_mean\',\'min\'),loss_max=(\'loss_mean\',\'max\'),\n        positive_losses=(\'loss_mean\',lambda s:int((s>0).sum()))).reset_index()\n    write_csv(root/\'rematching_sensitivity_summary.csv\',across)\n    n=len(parts);write_json(root/\'completion_summary.json\',dict(completed_runs=n,expected_runs=expected_runs,\n        evaluations=len(evaluations),complete=n==expected_runs,statistical_note=\'overlapping repeated constructions; no independent-cohort p-value or external-validation claim\'))\n    print(f\'Corridas completas: {n}/{expected_runs}; evaluaciones: {len(evaluations)}/{expected_runs*2}\')\n    return losses,cohort,across\n\n\ndef figures(root):\n    import matplotlib.pyplot as plt\n    root=Path(root);d=pd.read_csv(root/\'fixed_target_losses_by_cohort.csv\');d=d[d.n_seeds.eq(3)]\n    fig,axes=plt.subplots(1,2,figsize=(12,4.6),sharey=True)\n    colors=[\'#176B91\',\'#D87A18\',\'#25846B\',\'#AF475F\'];keys=list(MODELS)\n    for ax,target in zip(axes,DOMAINS):\n        for i,key in enumerate(keys):\n            g=d[d.target.eq(target)&d.model_key.eq(key)].sort_values(\'match_seed\')\n            if len(g):ax.scatter(i+np.linspace(-.13,.13,len(g)),g.loss_mean,s=46,color=colors[i],zorder=3)\n        ax.axhline(0,color=\'.35\',ls=\'--\',lw=1);ax.set_xticks(range(4),[\'ResNet-50\',\'ConvNeXt-T\',\'ViT-S/16\',\'Swin-T\'],rotation=20)\n        ax.set_title(\'Target \'+target);ax.grid(axis=\'y\',alpha=.2)\n    axes[0].set_ylabel(\'Fixed-target macro-F1 loss\\n(control − transfer; mean of 3 seeds)\')\n    fig.suptitle(\'P5 repeated matching: each dot is one construction\');fig.tight_layout()\n    for suffix in [\'png\',\'pdf\']:fig.savefig(root/f\'15_rematching_losses.{suffix}\',dpi=300,bbox_inches=\'tight\')\n    plt.show();plt.close(fig)\n    for metric in [\'val_macro_f1\',\'train_loss\']:\n        fig,axes=plt.subplots(2,4,figsize=(14,6),sharex=True)\n        for ax,(target,key) in zip(axes.flat,itertools.product(DOMAINS,keys)):\n            for match in MATCH_SEEDS:\n                hs=[]\n                for seed in TRAIN_SEEDS:\n                    p=root/\'runs\'/f\'match_{match}\'/f\'train_{target}__{key}__seed_{seed}\'/\'history.csv\'\n                    if p.exists():hs.append(pd.read_csv(p).set_index(\'epoch\')[metric])\n                if hs:\n                    mean=pd.concat(hs,axis=1).mean(axis=1);ax.plot(mean.index,mean,label=str(match),lw=1.3)\n            ax.set_title(f\'{target} | {key}\',fontsize=9);ax.grid(alpha=.2)\n            if metric==\'val_macro_f1\':ax.set_ylim(0,1.02)\n        axes[0,0].legend(fontsize=7,title=\'Matching seed\');fig.supxlabel(\'Epoch\');fig.supylabel(metric+\' (seed mean)\');fig.tight_layout()\n        for suffix in [\'png\',\'pdf\']:fig.savefig(root/f\'15_{metric}.{suffix}\',dpi=300,bbox_inches=\'tight\')\n        plt.show();plt.close(fig)\n\n\ndef run_all(meta_path,design_root,root,model_root,image_root,local_cache,experiment_name,workers=2,reference=None):\n    load_torch()\n    if not torch.cuda.is_available():raise RuntimeError(\'Activa una GPU en Colab antes de entrenar\')\n    if timm.__version__!=\'1.0.28\':raise RuntimeError(\'Se requiere timm==1.0.28; reinicia el runtime si se importó otra versión\')\n    root=Path(root);root.mkdir(parents=True,exist_ok=True);model_root=Path(model_root)\n    env=environment();freeze_json(root/\'environment_lock.json\',env)\n    meta=load_metadata(meta_path);cohorts=prepare_design(meta,design_root,reference=reference)\n    source_hash=sha(Path(__file__))\n    weight_cache=model_root/\'pretrained_cache\';ensure_pretrained(weight_cache)\n    weights={key:json.loads((weight_cache/(key+\'.json\')).read_text()) for key in MODELS}\n    plan=dict(version=VERSION,experiment_name=experiment_name,recipe=RECIPE,models=MODELS,\n              class_names=CLASSES,match_seeds=MATCH_SEEDS,train_seeds=TRAIN_SEEDS,metadata_sha256=sha(meta_path),source_sha256=source_hash,\n              assignments={str(s):sha(Path(design_root)/f\'match_{s}\'/\'assignments.csv\') for s in MATCH_SEEDS},environment=env,\n              num_workers=int(workers),pretrained_weights=weights)\n    freeze_json(root/\'experiment_plan.json\',plan)\n    signature=hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()\n    print(\'Verificando y copiando las imágenes al disco local...\',flush=True)\n    lookup,image_manifest=prepare_images(meta,cohorts,image_root,local_cache)\n    write_csv(root/\'verified_image_manifest.csv\',image_manifest)\n    atomic_bytes(root/\'executed_source.py\',Path(__file__).read_bytes())\n    tasks=list(itertools.product(MATCH_SEEDS,MODELS,TRAIN_SEEDS,DOMAINS));start=time.perf_counter();new_durations=[]\n    for i,(match,key,seed,domain) in enumerate(tasks,1):\n        print(f\'\\n[{i}/{len(tasks)}] matching={match} | {key} | seed={seed} | source={domain}\',flush=True)\n        before=time.perf_counter()\n        marker=root/\'runs\'/f\'match_{match}\'/f\'train_{domain}__{key}__seed_{seed}\'/\'COMPLETE.json\';already=marker.exists()\n        run_one(cohorts[match],match,key,seed,domain,lookup,root,model_root,weight_cache,signature,workers)\n        if not already:new_durations.append(time.perf_counter()-before)\n        if new_durations:\n            pending=sum(not (root/\'runs\'/f\'match_{m}\'/f\'train_{d}__{k}__seed_{s}\'/\'COMPLETE.json\').exists() for m,k,s,d in tasks)\n            print(f\'ETA orientativa restante: {pending*np.median(new_durations)/3600:.2f} h (incluye guardado observado)\',flush=True)\n        if i%6==0:\n            try:summarize(root)\n            except RuntimeError as e:print(\'Resumen parcial:\',e)\n    result=summarize(root);figures(root)\n    print(f\'FIN. Tiempo de esta sesión: {(time.perf_counter()-start)/3600:.2f} h. Resultados: {root}\')\n    return result\n\n\ndef export_results(root,design_root,exports_dir,experiment_name):\n    from zipfile import ZipFile,ZIP_DEFLATED\n    root=Path(root);design_root=Path(design_root);exports_dir=Path(exports_dir);exports_dir.mkdir(parents=True,exist_ok=True)\n    status=json.loads((root/\'completion_summary.json\').read_text())\n    if not status[\'complete\']:raise RuntimeError(\'El experimento aún no tiene las 120 corridas; se conserva el resumen parcial\')\n    destination=exports_dir/(experiment_name+\'_RESULTADOS.zip\');tmp=destination.with_suffix(\'.tmp.zip\')\n    with ZipFile(tmp,\'w\',ZIP_DEFLATED) as z:\n        for folder,prefix in [(root,\'results\'),(design_root,\'design\')]:\n            for p in sorted(folder.rglob(\'*\')):\n                if p.is_file() and not p.name.endswith(\'.tmp\'):z.write(p,Path(prefix)/p.relative_to(folder))\n        z.writestr(\'LEEME.txt\',\'120 entrenamientos nuevos, 240 evaluaciones. Pesos y checkpoints permanecen aparte en models/.\\nNo incluye imágenes ni credenciales. Las construcciones se solapan: no son cohortes de adquisición independientes.\\n\')\n    with ZipFile(tmp) as z:\n        if z.testzip() is not None:raise RuntimeError(\'ZIP incompleto\')\n    os.replace(tmp,destination);print(\'ZIP para devolver al análisis:\',destination)\n    return destination\n'
METADATA_COMPRESSED = '...'
REFERENCE_COMPRESSED = '...'
METADATA_SHA256 = '62712567a3b81d56321c90075dc6558e4b4c3d536a2e08aae457c8a0fee293e5'
REFERENCE_SHA256 = '384f6f5495639d03677cc41882cff72c9ac3204b2dc463e08b2cb3e2fce82788'

module_path = Path('/content/paddy15_rematching.py')
module_path.write_text(MODULE_TEXT, encoding='utf-8')
spec = importlib.util.spec_from_file_location('paddy15_rematching', module_path)
p15 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = p15
spec.loader.exec_module(p15)

META_PATH = DESIGN_ROOT / '03_paddy_doctor_nearclean_frozen.csv'
REFERENCE_PATH = DESIGN_ROOT / 'P5_original_assignments_reference.csv'
for path, encoded, expected in [(META_PATH, METADATA_COMPRESSED, METADATA_SHA256),
                                 (REFERENCE_PATH, REFERENCE_COMPRESSED, REFERENCE_SHA256)]:
    payload = zlib.decompress(base64.b64decode(encoded))
    assert hashlib.sha256(payload).hexdigest() == expected
    if path.exists() and p15.sha(path) != expected:
        raise RuntimeError(f'Cambió un archivo congelado: {path}')
    if not path.exists():
        p15.atomic_bytes(path, payload)

import pandas as pd
metadata = p15.load_metadata(META_PATH)
reference = pd.read_csv(REFERENCE_PATH, dtype={'image_id':str})
print('Metadata near-clean:', len(metadata), 'imágenes')
print('Clases P5:', p15.CLASSES)
print('Semillas de emparejamiento:', p15.MATCH_SEEDS)
print('Semillas de entrenamiento:', p15.TRAIN_SEEDS)


## 5. Construir y verificar los cinco emparejamientos
Por clase se forman parejas ADT45/OTHER con **la misma edad registrada**, sin reemplazo dentro de una construcción; se seleccionan 100 pares y se asignan 60/20/20 a entrenamiento/validación/prueba. Las dos imágenes del par van a la misma partición. Cada dominio tiene 300/100/100 imágenes.

Se comprueban IDs, SHA-256, grupos visuales disponibles y distribución exacta de edades entre dominios dentro de cada clase y partición. Todas las arquitecturas reutilizan estas asignaciones. Se mide el solapamiento entre construcciones, con el P5 original y entre sus particiones. Una imagen que pasa de entrenamiento en una construcción a prueba en otra **no se evalúa con el modelo entrenado en la primera**: cada modelo pertenece solo a su propia construcción.


In [ ]:
cohorts = p15.prepare_design(metadata, DESIGN_ROOT, reference=reference)
display(pd.read_csv(DESIGN_ROOT / 'design_manifest.csv'))
display(pd.read_csv(DESIGN_ROOT / 'overlap_between_cohorts.csv').query("partition == 'all'"))
print('CONTROLES APROBADOS: 5 construcciones × 1000 imágenes; 500 pares en cada una.')


## 6. Encontrar o descargar las imágenes
Se buscan las ubicaciones habituales de tu proyecto y la caché de Kaggle. Si faltan, se usa la credencial existente `config/kaggle.json`. Solo si no está disponible se pedirá subirla; se usa en el runtime y no se incluye en resultados. La cuenta debe haber aceptado las condiciones de la competición.

No hace falta repetir los notebooks de auditoría ni descargar un conjunto distinto. Antes de entrenar se verificarán los bytes de cada imagen seleccionada frente al SHA-256 histórico y se copiarán a `/content` para evitar lectura por lote desde Drive.


In [ ]:
def train_images_in(path):
    path = Path(path)
    if path.name == 'train_images' and path.is_dir(): return path
    if (path / 'train_images').is_dir(): return path / 'train_images'
    return None

candidates = ([Path(DATASET_ROOT)] if DATASET_ROOT else []) + [
    PROJECT / 'data/raw/paddy_doctor', PROJECT / 'data/raw', PROJECT / 'data',
    Path('/root/.cache/kagglehub/competitions/paddy-disease-classification'),
    Path('/content/paddy-disease-classification')]
IMAGE_ROOT = next((found for c in candidates if (found := train_images_in(c)) is not None), None)
if DATASET_ROOT and train_images_in(DATASET_ROOT) is None:
    raise FileNotFoundError('DATASET_ROOT no contiene train_images. Corrige la ruta.')
if IMAGE_ROOT is None:
    credential = PROJECT / 'config/kaggle.json'
    destination = Path.home() / '.kaggle/kaggle.json'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if credential.exists():
        shutil.copyfile(credential, destination); os.chmod(destination, 0o600)
    elif not destination.exists() and not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import files
        print('Selecciona kaggle.json (no se imprime su contenido).')
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded: raise RuntimeError('Falta kaggle.json')
        destination.write_bytes(uploaded['kaggle.json']); os.chmod(destination, 0o600)
        del uploaded
    import kagglehub
    downloaded = Path(kagglehub.competition_download('paddy-disease-classification'))
    IMAGE_ROOT = train_images_in(downloaded)
    if IMAGE_ROOT is None:
        options = list(downloaded.rglob('train_images'))
        if len(options) == 1: IMAGE_ROOT = options[0]
    if IMAGE_ROOT is None: raise FileNotFoundError('La descarga no contiene train_images')
print('Carpeta de imágenes:', IMAGE_ROOT)


## 7. Verificar GPU y pesos antes del bloque largo
Se usan etiquetas explícitas del registro auditado de `timm 1.0.28`. Las cuatro configuraciones tienen procedencias de preentrenamiento diferentes; esto **no** constituye una comparación que aísle solamente la arquitectura.

Se descargan una sola vez, se guarda su configuración y se calculan huellas del archivo y los tensores. Cada corrida reemplaza la cabeza de 1000 clases por una de cinco, con semilla registrada, y guarda la huella de su inicialización efectiva. Si un peso no está disponible, el cuaderno se detiene; no lo reemplaza por otra variante.

Los checkpoints completos pueden ocupar alrededor de 12 GB. La comprobación de espacio local que sigue no informa la cuota restante de tu cuenta Drive.


In [ ]:
p15.load_torch()
if not p15.torch.cuda.is_available():
    raise RuntimeError('Activa GPU en Entorno de ejecución → Cambiar tipo de entorno')
if p15.timm.__version__ != '1.0.28':
    raise RuntimeError('Reinicia la sesión: hay otra versión de timm cargada en memoria')
print(json.dumps(p15.environment(), indent=2))
print('Espacio local libre:', round(shutil.disk_usage('/content').free / 1024**3, 1), 'GB')
print('Comprueba también que tu cuenta Drive tenga unos 15 GB disponibles.')
p15.ensure_pretrained(MODEL_ROOT / 'pretrained_cache')


## 8. Entrenar las 120 corridas y evaluar
**Esta es la celda larga.** Conserva 30 épocas, AdamW, batch 32, resolución 224, calentamiento de dos épocas y coseno. El recorte tiene explícitamente ratio `(0.75, 4/3)`, como el valor por defecto usado por el código P5 revisado. La selección del mejor checkpoint usa únicamente macro-F1 de validación del dominio de entrenamiento. Los tests se evalúan al terminar el entrenamiento; no eligen época ni parámetros.

La aleatoriedad del cargador se fija por época y se recrean sus workers. Es una ejecución nueva y trazable; no una reconstrucción byte a byte del entrenamiento histórico. Se registra el entorno, el código, la configuración y los pesos.

Por corrida se guardan: historial, mejor checkpoint, logits de validación y de ambos tests, predicciones por imagen, F1 por clase, matrices de confusión, F1 macro, exactitud balanceada, ECE uniforme/adaptativo, MCE, Brier y NLL. También se ajusta una temperatura solo con validación fuente y se conserva su diagnóstico. Esa validación se reutiliza para selección y calibración: no se afirma una garantía formal.

**Reanudación:** solo se omiten corridas con `COMPLETE.json` cuyos archivos y checkpoint coinciden con sus huellas. Si se corta una corrida de 30 épocas, esa corrida empieza de nuevo. El historial parcial se reemplaza al repetirla. No cambies el diseño o las versiones durante la ejecución.


In [ ]:
losses, by_cohort, sensitivity = p15.run_all(
    meta_path=META_PATH, design_root=DESIGN_ROOT,
    root=RUN_ROOT, model_root=MODEL_ROOT,
    image_root=IMAGE_ROOT,
    local_cache=Path('/content/paddy15_verified_images'),
    experiment_name=EXPERIMENT_NAME, workers=NUM_WORKERS,
    reference=reference,
)


## 9. Revisar el resultado central
La pérdida se calcula **sobre las mismas imágenes objetivo**: F1 del modelo entrenado en ese dominio menos F1 del modelo entrenado en el dominio opuesto, manteniendo arquitectura y semilla. No se resta el control de ADT45 al test de OTHER.

La tabla por construcción promedia tres semillas. La tabla final muestra media, desviación y rango entre cinco construcciones, además del número con pérdida positiva. Son sensibilidades descriptivas con solapamiento; no se tratan cinco construcciones como cinco datasets independientes ni se genera automáticamente un p-valor basado en esa suposición. El experimento puede confirmar, matizar o contradecir la conclusión actual.


In [ ]:
losses, by_cohort, sensitivity = p15.summarize(RUN_ROOT)
display(sensitivity)
status = json.loads((RUN_ROOT / 'completion_summary.json').read_text())
assert status['complete'] and status['completed_runs'] == 120 and status['evaluations'] == 240
print('EXPERIMENTO COMPLETO: 120 entrenamientos y 240 evaluaciones.')


## 10. Crear el ZIP para actualizar el artículo
Este ZIP incluye asignaciones, solapamiento, código ejecutado, entorno, procedencia de pesos, métricas, logits, predicciones y figuras. **No incluye** imágenes, credenciales ni los 120 checkpoints; estos permanecen en `models/`.

Cuando termine, el archivo que necesitas devolver para actualizar el manuscrito es `15_P5_Rematching_5x3_30ep_v1_RESULTADOS.zip`, en `PADDY_DOCTOR_Q1/exports/`.


In [ ]:
zip_path = p15.export_results(RUN_ROOT, DESIGN_ROOT, EXPORTS_DIR, EXPERIMENT_NAME)
print('Resultados guardados:', zip_path)
print('Tamaño ZIP:', round(zip_path.stat().st_size / 1024**2, 2), 'MB')
# Descarga opcional al ordenador: descomenta las dos líneas siguientes.
# from google.colab import files
# files.download(str(zip_path))


### Procedencia y límites
Metadatos near-clean y asignaciones originales proceden del paquete de revisión v3 del proyecto. Se mantiene la exclusión de duplicados auditada, sin afirmar independencia por planta, finca o sesión. No se selecciona el mejor emparejamiento a partir del test; deben conservarse los cinco.

No se pueden combinar estas predicciones como observaciones independientes con las del P5 histórico. Los resultados nuevos se analizarán como una sensibilidad repetida, con sus configuraciones explícitas. Las discrepancias históricas de trazabilidad de pesos no se borran por generar pesos nuevos trazables.

Documentación técnica consultada: [timm Model API](https://huggingface.co/docs/timm/models) · [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html) · [PyTorch checkpoints](https://docs.pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html).
